# 02. Faster R-CNN (двостадійний, з anchors)

Протокол один для всіх torchvision-моделей:
- дані: `prepared/train` (кожен 3-й кадр, 960 px, ignore-зони сірі), валідація на `prepared/val` (7 послідовностей, кожен 2-й кадр);
- модель: ResNet50-FPN з вагами COCO, без внутрішнього ресайзу (`min_size=540, max_size=960`);
- навчання: SGD, momentum 0.9, weight decay 1e-4, batch 8, lr 0.01, warmup 300 ітерацій, 6 епох, lr ×0.1 після 4-ї і 5-ї епохи, AMP, горизонтальний фліп;
- оцінка: прогнози в оригінальних координатах, `common.evaluate` (mAP, AP50, AP_small; ignore-регіони як `iscrowd`), до 500 детекцій на кадр;
- швидкість: batch 1, FP32, разом з постобробкою, мс на кадр 960×540.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

import analysis as an
import common as c
import tv_models as tv

c.set_seed()
P = c.paths()
OUT = P.out / "two_stage"
print("відновлено з Input:", c.restore_from_inputs(P, "two_stage"))
GT = P.prep / "val" / "annotations_eval.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ANCHORS = tv.anchors_from_kmeans(P.prep / "anchors_960.json")
PROTOCOL = dict(epochs=6, batch=8, lr=0.01, milestones=(4, 5), warmup=300, workers=4)
if DEBUG:
    PROTOCOL.update(epochs=1, batch=2, warmup=5, workers=0, milestones=(1,))
print(P.prep, DEVICE, "anchors:", ANCHORS)
PROTOCOL

## 1. Експерименти

Кожен наступний експеримент міняє один параметр, щоб було видно, що саме дало приріст.

In [ ]:
EXPERIMENTS = {

    "frcnn_base": dict(),
    "frcnn_anchors": dict(anchor_base=ANCHORS),
    "frcnn_anchors_roi": dict(anchor_base=ANCHORS, box_positive_fraction=0.5),
    "frcnn_anchors_rpn": dict(anchor_base=ANCHORS, rpn_pre_nms_top_n_train=4000, rpn_post_nms_top_n_train=4000),
}
RUN = ["frcnn_base", "frcnn_anchors", "frcnn_anchors_roi"]


def load_model(name):
    model = tv.build_model("faster_rcnn", pretrained=False, **EXPERIMENTS[name])
    model.load_state_dict(torch.load(OUT / name / "best.pt", map_location="cpu"))
    return model.to(DEVICE).eval()


def run_experiment(name):
    model = tv.build_model("faster_rcnn", pretrained=not DEBUG, **EXPERIMENTS[name])
    hist, _ = tv.train(model, P.prep, OUT / name, train_every=20 if DEBUG else 1, **PROTOCOL)
    model = load_model(name)
    dets = tv.predict(model, P.prep, "val", workers=PROTOCOL["workers"])
    c.save_json(dets, OUT / name / "val_dets.json")
    metrics = c.evaluate(GT, dets, per_class=True)
    ms = tv.measure_speed(model, P.prep, n=5 if DEBUG else 100)
    return an.log_result(P.results, name, "two-stage", metrics, ms_per_img=ms, epochs=len(hist),
                         gpu_hours=sum(h["train_min"] for h in hist) / 60, config=str(EXPERIMENTS[name]))


for name in RUN:
    print("=" * 20, name)
    run_experiment(name)
    torch.cuda.empty_cache()

## 2. Результати на val

In [ ]:
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "gpu_hours"]
res = an.all_results(P)
res = res[res.family == "two-stage"].set_index("name")
display(res[COLS].round(3))
display(res[[f"AP_{k}" for k in c.CLASSES]].round(3))
BEST = res.loc[[n for n in RUN if n in res.index], "mAP"].idxmax()
print("найкраща конфігурація:", BEST)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for name in RUN:
    h = pd.DataFrame(c.load_json(OUT / name / "history.json"))
    loss = h.filter(like="loss").sum(axis=1)
    axes[0].plot(h.epoch, loss, marker="o", label=name)
    axes[1].plot(h.epoch, h.mAP, marker="o", label=name)
    axes[2].plot(h.epoch, h.AP_small, marker="o", label=name)
for ax, t in zip(axes, ["сумарний train loss", "val mAP", "val AP_small"]):
    ax.set_title(t)
    ax.set_xlabel("епоха")
axes[1].legend()
fig.savefig(P.images / "frcnn_curves.png", bbox_inches="tight", dpi=130)
plt.show()

## 3. Кількість RPN-пропозицій при інференсі

Перенавчання не потрібне: беремо найкращу модель і міняємо кількість пропозицій після NMS.

In [ ]:
model = load_model(BEST)
rows = {}
for n in (300, 1000, 2000, 3000):
    tv.set_proposals(model, n, n)
    dets = tv.predict(model, P.prep, "val", workers=PROTOCOL["workers"])
    r = c.evaluate(GT, dets)
    rows[n] = {k: r[k] for k in ("mAP", "AP50", "AP_small", "AR_max")}
    rows[n]["ms_per_img"] = tv.measure_speed(model, P.prep, n=5 if DEBUG else 50)
tv.set_proposals(model, 1000, 1000)
proposals = pd.DataFrame(rows).T.rename_axis("RPN post-NMS top-N")
c.save_json(proposals.reset_index().to_dict("records"), OUT / "proposals_sweep.json")
proposals.round(3)

## 4. Що дає кожна стадія

Порівнюємо recall пропозицій RPN (IoU ≥ 0.5 з будь-якою пропозицією, клас не важливий) з recall фінальних детекцій (правильний клас, IoU ≥ 0.5) на тих самих кадрах.

In [ ]:
EVERY = 5
rpn = pd.DataFrame(tv.rpn_recall(model, P.prep, top_ns=(100, 300, 1000, 2000), every=EVERY)).T
rpn.index = [f"RPN top-{n}" for n in rpn.index]
dets = c.load_json(OUT / BEST / "val_dets.json")
ids = {im["id"] for im in tv.PreparedDataset(P.prep, "val", every=EVERY).images}
RAW = P.data / "val" / "annotations_coco.json"
for thr in (0.05, 0.3):
    _, g_df, _ = an.error_breakdown(GT, dets, score_thr=thr)
    rpn.loc[f"фінальні детекції, score ≥ {thr}"] = g_df[g_df.image_id.isin(ids)].groupby("size").found.mean()
rpn[["small", "medium", "large"]].round(3)

## 5. Помилки

Типи детекцій при порозі 0.3 (спрощений TIDE, див. `analysis.error_breakdown`), recall за розміром і класом, матриця плутанини.

In [ ]:
d_df, g_df = an.error_report(GT, dets, P.data / "val", RAW, P.images, prefix="frcnn", score_thr=0.3)

## Висновки

Усі числа з таблиць цього ноутбука (val, `common.evaluate`); кожна конфігурація навчалася один раз, тому різницю до 0.01 mAP не варто вважати значущою. Найкраща епоха в усіх трьох конфігураціях перша: далі при lr 0.01 mAP не росте.

**Що дає кожна стадія.** RPN з top-1000 пропозицій знаходить (клас не важливий, IoU ≥ 0.5) 67.5% малих, 81.1% середніх і 98.7% великих об'єктів. Фінальні детекції (правильний клас, score ≥ 0.05) покривають 49.2%, 64.1% і 92.6%. Друга стадія тому губить 18 п. п. малих і 17 п. п. середніх, а великих лише 6 п. п. Стеля для малих об'єктів задана першою стадією: 67.5%, і top-2000 її не піднімає.

**Anchors.** k-means anchors майже не змінили mAP (0.175 → 0.174), але підняли AP50 (0.365 → 0.375), AP_small (0.062 → 0.077, +0.015) і AR_max (0.393 → 0.408). Виграли bicycle (0.114 → 0.137), car і person, програли крупніші van (0.211 → 0.178) і truck (0.251 → 0.227).

**Вибірка RoI.** `box_positive_fraction=0.5` замість 0.25 погіршила mAP до 0.163, AR_max до 0.352, а truck до 0.126. Імовірно, слабкі RoI з низьким IoU потрапляють у позитиви й шумлять у навчанні.

**Кількість пропозицій при інференсі.** Перехід 300 → 1000 дає mAP 0.164 → 0.175 і AP_small 0.047 → 0.062, далі приріст 0.001 при 2000 і нуль при 3000. Час росте: 89, 96, 109, 128 мс. Залишаємо 1000.

**Три типи збоїв** (score ≥ 0.3: tp 49.8%, cls 12.8%, loc 9.3%, bg 12.3%, ignore 10.5%, dup 1.4%):
1. Малі об'єкти: recall 0.38 проти 0.84 у великих. Причини: стеля RPN (67.5%) і мало пікселів після ресайзу до 960 px.
2. Рідкі й тонкі класи: bus має recall 0.008 (у val лише 132 об'єкти), bicycle 0.125. Імовірно, bus мало в train і він схожий на truck і van, а bicycle тонкий і зливається з людиною.
3. Перекриття: recall 0.715 без перекриття, 0.456 при частковому і 0.137 при сильному. Видно лише частину об'єкта, і RoI-ознаки слабкі.

Швидкість усіх трьох моделей ~97 мс на кадр (FP32, batch 1), навчання ~1.2 GPU-год.